# Golden Transcription Selection Pipeline v5 - Arabic_SA Submission

**Final submission notebook for the Transcription Assessment challenge (Arabic_SA dataset).**

## Architecture: Two-Regime Adaptive Ensemble

| Component | Detail |
|-----------|--------|
| **ASR Model 1** | Whisper large-v3 (1.55B params, auto language detect, beam search) |
| **ASR Model 2** | SeamlessM4T v2-large (~2.3B params, explicit `tgt_lang=arb`) |
| **Hard Filters** | Script detection (>1000 chars), speaker labels, stage directions, scene markers |
| **Scoring Signals** | ASR similarity, inter-option consensus, fluency, quality penalty, relative length |
| **Regime Detection** | Per-sample diversity -> similar (consensus-heavy) vs diverse (ASR-heavy) |
| **Optimization** | Grid search over weight space, LOO-CV validation |

### Key Insight (Arabic_SA)
All 49 labeled samples fall into the **similar regime** - options differ only in punctuation/formatting.  
The golden transcription is the "odd one out" with **negative consensus** (least similar to peers).  
**Expected accuracy: 95.9% (47/49) on labeled subset.**

### Dataset
- 100 Arabic_SA audio samples (49 labeled, 51 unlabeled)
- 5 candidate transcriptions per sample
- Audio served via CloudFront URLs

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
import subprocess, sys, shutil

# Import torch first (before pip touches anything)
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

# Install ffmpeg
result = subprocess.run(
    ["sudo", "apt-get", "install", "-y", "-qq", "ffmpeg"],
    capture_output=True, text=True
)
if result.returncode != 0:
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], capture_output=True)

# Install packages (without upgrading torch)
!pip install -q openai-whisper jiwer pandas tqdm requests pydub 2>/dev/null
!pip install -q transformers torchaudio sentencepiece protobuf --no-deps 2>/dev/null
!pip install -q sentencepiece protobuf 2>/dev/null

assert shutil.which("ffmpeg"), "ffmpeg not found!"
print(f"ffmpeg: {shutil.which('ffmpeg')}")
print("Dependencies ready.")

In [ ]:
# ============================================================
# Cell 2: Imports & Configuration
# ============================================================
import os, re, gc, glob, warnings
import pandas as pd
import numpy as np
import requests
import whisper
import torch
import torchaudio
from tqdm.auto import tqdm
from jiwer import wer, cer
from pathlib import Path
from itertools import product as iter_product
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")

# ---- CONFIG ----
OUTPUT_CSV = "/kaggle/working/golden_transcriptions_output.csv"
WHISPER_MODEL = "large-v3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DIVERSITY_THRESHOLD = 0.3  # Regime split threshold

# ---- Auto-detect dataset path on Kaggle ----
CSV_PATH = None
for pattern in [
    "/kaggle/input/*/Transcription Assessment Arabic_SA Dataset*.csv",
    "/kaggle/input/*/transcription*arabic*.csv",
    "/kaggle/input/*/*.csv",
]:
    matches = sorted(glob.glob(pattern))
    if matches:
        CSV_PATH = matches[0]
        break

if CSV_PATH is None:
    CSV_PATH = "/kaggle/input/hackenza-3/Transcription Assessment Arabic_SA Dataset (1).csv"

print(f"Device:    {DEVICE}")
print(f"Whisper:   {WHISPER_MODEL}")
print(f"CSV:       {CSV_PATH}")
print(f"CSV exists: {os.path.exists(CSV_PATH)}")

In [ ]:
# ============================================================
# Cell 3: Load Dataset
# ============================================================
df = pd.read_csv(CSV_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Language: {df['language'].unique()}")
print(f"Labeled: {df['correct_option'].notna().sum()}, Unlabeled: {df['correct_option'].isna().sum()}")

# Parse correct_option to integer
def parse_correct_option(val):
    """Parse correct_option to integer. Handles 'option_1', '1', 1.0 formats."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.startswith('option_'):
        return int(s.replace('option_', ''))
    try:
        return int(float(s))
    except ValueError:
        return np.nan

df['correct_option_int'] = df['correct_option'].apply(parse_correct_option)
print(f"Correct option distribution: {df['correct_option_int'].dropna().astype(int).value_counts().sort_index().to_dict()}")
df.head(3)

In [ ]:
# ============================================================
# Cell 4: Download Audio Files
# ============================================================
AUDIO_DIR = "/kaggle/working/audio_files"
os.makedirs(AUDIO_DIR, exist_ok=True)

def download_audio(row):
    """Download audio from URL. Returns (audio_id, filepath, success)."""
    audio_id = row['audio_id']
    url = str(row['audio'])
    filepath = os.path.join(AUDIO_DIR, f"audio_{audio_id}.wav")
    if os.path.exists(filepath):
        return (audio_id, filepath, True)
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        with open(filepath, 'wb') as f:
            f.write(resp.content)
        return (audio_id, filepath, True)
    except Exception as e:
        print(f"  FAILED audio_id={audio_id}: {e}")
        return (audio_id, filepath, False)

print(f"Downloading {len(df)} audio files...")
audio_paths = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(download_audio, row): row['audio_id']
               for _, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
        audio_id, filepath, success = future.result()
        if success:
            audio_paths[audio_id] = filepath

df['audio_path'] = df['audio_id'].map(audio_paths)
existing = df['audio_path'].notna().sum()
print(f"Audio files ready: {existing}/{len(df)}")

In [ ]:
# ============================================================
# Cell 5: Whisper large-v3 Transcription
# ============================================================
print(f"Loading Whisper {WHISPER_MODEL} on {DEVICE}...")
whisper_model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
print("Whisper model loaded!")

def transcribe_whisper(filepath):
    """Transcribe audio with Whisper large-v3."""
    try:
        result = whisper_model.transcribe(
            filepath,
            language=None,       # auto-detect
            task="transcribe",
            beam_size=5,
            best_of=5,
            temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            condition_on_previous_text=True,
            no_speech_threshold=0.6,
            compression_ratio_threshold=2.4,
        )
        return result['text'].strip(), result.get('language', 'unknown')
    except Exception as e:
        print(f"  Whisper error: {e}")
        return "", "error"

print("\nTranscribing with Whisper large-v3...")
whisper_texts = {}
detected_langs = {}

for _, row in tqdm(df.iterrows(), total=len(df), desc="Whisper"):
    aid = row['audio_id']
    fp = row.get('audio_path', '')
    if fp and os.path.exists(str(fp)):
        text, lang = transcribe_whisper(fp)
        whisper_texts[aid] = text
        detected_langs[aid] = lang
    else:
        whisper_texts[aid] = ""
        detected_langs[aid] = "missing"

df['whisper_text'] = df['audio_id'].map(whisper_texts)
df['detected_lang'] = df['audio_id'].map(detected_langs)
print(f"Whisper done! Detected languages: {pd.Series(detected_langs).value_counts().to_dict()}")
print(f"Sample: {list(whisper_texts.values())[0][:200]}")

# Free GPU memory for SeamlessM4T
del whisper_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Whisper model unloaded.")

In [ ]:
# ============================================================
# Cell 6: SeamlessM4T v2-large Transcription
# ============================================================
print("Loading SeamlessM4T v2-large...")
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

seamless_processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
seamless_model = SeamlessM4Tv2ForSpeechToText.from_pretrained("facebook/seamless-m4t-v2-large")
seamless_model = seamless_model.to(DEVICE)
seamless_model.eval()
print("SeamlessM4T loaded!")

# Language mapping
lang_to_seamless = {
    'arabic': 'arb', 'Arabic_SA': 'arb', 'arb': 'arb',
    'english': 'eng', 'English': 'eng', 'eng': 'eng',
    'hindi': 'hin', 'Hindi': 'hin', 'hin': 'hin',
    'french': 'fra', 'spanish': 'spa', 'german': 'deu',
}

def transcribe_seamless(filepath, lang_hint='Arabic_SA'):
    """Transcribe audio with SeamlessM4T v2."""
    try:
        waveform, sr = torchaudio.load(filepath)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        inputs = seamless_processor(
            audio=waveform.squeeze(0).numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        ).to(DEVICE)
        
        tgt = lang_to_seamless.get(lang_hint, 'arb')  # Default Arabic for this dataset
        
        with torch.no_grad():
            output_tokens = seamless_model.generate(**inputs, tgt_lang=tgt)
        text = seamless_processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)
        return text.strip()
    except Exception as e:
        print(f"  SeamlessM4T error for {filepath}: {e}")
        return ""

print("\nTranscribing with SeamlessM4T v2...")
seamless_texts = {}

for _, row in tqdm(df.iterrows(), total=len(df), desc="SeamlessM4T"):
    aid = row['audio_id']
    fp = row.get('audio_path', '')
    lang = row.get('language', 'Arabic_SA')
    if fp and os.path.exists(str(fp)):
        seamless_texts[aid] = transcribe_seamless(fp, lang_hint=lang)
    else:
        seamless_texts[aid] = ""

df['seamless_text'] = df['audio_id'].map(seamless_texts)
print(f"SeamlessM4T done!")
print(f"Sample: {list(seamless_texts.values())[0][:200]}")

# Free GPU memory
del seamless_model, seamless_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("SeamlessM4T model unloaded.")

In [ ]:
# ============================================================
# Cell 7: Text Normalization (Arabic-optimized)
# ============================================================

def normalize_arabic(text):
    """Normalize Arabic text for fair WER comparison."""
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652\u0670]', '', text)  # Remove diacritics
    text = re.sub(r'[\u0625\u0623\u0622\u0671]', '\u0627', text)   # Normalize alef variants
    text = text.replace('\u0629', '\u0647')                          # taa marbuta -> haa
    text = text.replace('\u0649', '\u064A')                          # alef maksura -> yaa
    text = text.replace('\u0640', '')                                 # Remove tatweel
    return text

def normalize_text(text):
    """Universal text normalization."""
    if not text or not isinstance(text, str):
        return ""
    text = str(text).strip()
    text = re.sub(r'\([^)]*\)', ' ', text)               # Remove stage directions
    text = re.sub(r'\[[^\]]*:\]', ' ', text)             # Remove [Speaker:]
    text = re.sub(r'\[[^\]]*\]', ' ', text)              # Remove [annotations]
    text = re.sub(r'\u0627\u0644\u0645\u0634\u0647\u062F\s*[\u0660-\u0669\d]+', ' ', text)  # Scene markers
    text = re.sub(r'\u0627\u0644\u0645\u0643\u0627\u0646\s*:', ' ', text)  # Location markers
    text = re.sub(r'Scene\s*\d+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[\[\{][^}\]]*[\]\}]', ' ', text)    # Remove {silence} etc.
    text = re.sub(r'[\u061F!\u060C\u061B:.\u2026\-\u2013\u2014\"\'\'\'\`\(\)\[\]\{\}\u00AB\u00BB\u201C\u201D\u2018\u2019,;!?\\\\/]', ' ', text)
    text = normalize_arabic(text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

# Quick test
for t in ['[\u062B\u064A\u0648:] \u0647\u0647\u0647\u0647 \u0645\u0628\u0631\u0648\u0643',
          '\u0644\u064A\u0646\u0627: (\u062A\u062A\u0646\u0647\u062F) \u062B\u064A\u0648!',
          '\u0627\u0627\u0627\u0627\u0647 \u062B\u064A\u0648']:
    print(f"  '{t}' -> '{normalize_text(t)}'")

In [ ]:
# ============================================================
# Cell 8: Hard Quality Filters & Feature Engineering
# ============================================================
# Golden transcriptions are ALWAYS pure verbatim text.
# They NEVER contain speaker labels, stage directions, or scene markers.

def is_script_option(text):
    """Detect screenplay scripts (>1000 chars)."""
    return isinstance(text, str) and len(text) > 1000

def has_speaker_labels(text):
    if not isinstance(text, str): return False
    if re.search(r'\[[^\]]*:\]', text): return True
    if re.search(r'(?:^|[\n])[\u0600-\u06FF]+\s*:', text): return True
    return False

def has_stage_directions(text):
    if not isinstance(text, str): return False
    return bool(re.search(r'\([^)]{3,}\)', text))

def has_scene_markers(text):
    if not isinstance(text, str): return False
    return any(m in text for m in ['\u0627\u0644\u0645\u0634\u0647\u062F', '\u0627\u0644\u0645\u0643\u0627\u0646:', 'Scene '])

def compute_quality_penalty(text):
    """Quality penalty: higher = more annotation artifacts = less likely golden."""
    if not isinstance(text, str) or len(text.strip()) < 3:
        return 1.0
    penalty = 0.0
    penalty += len(re.findall(r'\[[^\]]*:\]', text)) * 0.15
    penalty += len(re.findall(r'\([^)]{3,}\)', text)) * 0.12
    penalty += len(re.findall(r'[\u0600-\u06FF]+\s*:', text)) * 0.05
    if has_scene_markers(text): penalty += 0.2
    penalty += text.count('{') * 0.02
    return min(penalty, 1.0)

# Validate: golden options should NEVER be filtered
labeled = df.dropna(subset=['correct_option_int']).copy()
labeled['correct_int'] = labeled['correct_option_int'].astype(int)
n_labeled = len(labeled)

filter_safe = 0
for _, row in labeled.iterrows():
    text = str(row[f"option_{int(row['correct_int'])}"])
    if not any([is_script_option(text), has_speaker_labels(text),
                has_stage_directions(text), has_scene_markers(text)]):
        filter_safe += 1

print(f"Hard filter safety check: {filter_safe}/{n_labeled} golden options pass all filters")
assert filter_safe == n_labeled, "DANGER: Hard filter would eliminate a golden option!"

In [ ]:
# ============================================================
# Cell 9: Compute All Scoring Signals
# ============================================================

def compute_wer_safe(ref, hyp):
    r, h = normalize_text(ref), normalize_text(hyp)
    if not r or not h: return 1.0
    if r == h: return 0.0
    try: return min(wer(r, h), 1.0)
    except: return 1.0

def compute_similarity(ref, hyp):
    return 1.0 - compute_wer_safe(ref, hyp)

print(f"Computing scoring signals for {len(df)} samples x 5 options...")
signal_data = {}  # signal_data[audio_id][option_num] = dict of signals

for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    aid = row['audio_id']
    w_ref = whisper_texts.get(aid, "")
    s_ref = seamless_texts.get(aid, "")
    
    options = {j: str(row[f'option_{j}']) for j in range(1, 6)}
    valid = {j: t for j, t in options.items()
             if not is_script_option(t) and len(t.strip()) >= 5}
    
    row_signals = {}
    for j, text in options.items():
        sig = {}
        
        # Mark scripts/empty as filtered
        if is_script_option(text) or len(text.strip()) < 5:
            sig.update({'is_script': True, 'hard_filter': True,
                        'whisper_sim': 0.0, 'seamless_sim': 0.0,
                        'consensus': 0.0, 'fluency': 0.0,
                        'quality_penalty': 1.0})
            row_signals[j] = sig
            continue
        
        sig['is_script'] = False
        sig['has_speaker'] = has_speaker_labels(text)
        sig['has_stage'] = has_stage_directions(text)
        sig['has_scene'] = has_scene_markers(text)
        sig['hard_filter'] = sig['has_speaker'] or sig['has_stage'] or sig['has_scene']
        
        # Signal 1: Whisper similarity
        sig['whisper_sim'] = compute_similarity(w_ref, text) if w_ref else 0.5
        
        # Signal 2: SeamlessM4T similarity
        sig['seamless_sim'] = compute_similarity(s_ref, text) if s_ref else 0.5
        
        # Signal 3: Consensus (avg pairwise similarity with other valid options)
        pairwise = [compute_similarity(text, other)
                    for k, other in valid.items() if k != j]
        sig['consensus'] = np.mean(pairwise) if pairwise else 0.5
        
        # Signal 4: Fluency proxy
        words = normalize_text(text).split()
        sig['fluency'] = len(words) / max(len(normalize_text(text)), 1) if words else 0.0
        
        # Signal 5: Quality penalty
        sig['quality_penalty'] = compute_quality_penalty(text)
        
        # Signal 6: Text length (for relative length normalization)
        sig['text_length'] = len(text)
        
        row_signals[j] = sig
    
    # Normalize relative length
    valid_lengths = [row_signals[j].get('text_length', 0) for j in valid]
    if valid_lengths:
        max_len = max(valid_lengths) if max(valid_lengths) > 0 else 1
        for j in valid:
            row_signals[j]['rel_length'] = row_signals[j].get('text_length', 0) / max_len
    
    signal_data[aid] = row_signals

# Stats
pass_counts = [sum(1 for s in sigs.values() if not s.get('hard_filter', True))
               for sigs in signal_data.values()]
print(f"\nScoring complete for {len(signal_data)} samples.")
print(f"Options passing hard filter: mean={np.mean(pass_counts):.1f}, min={min(pass_counts)}, max={max(pass_counts)}")

In [ ]:
# ============================================================
# Cell 10: Regime Detection
# ============================================================
# Diversity = 1 - avg_consensus. Low = similar options, High = diverse.

def compute_sample_diversity(row_signals):
    valid_keys = [j for j, s in row_signals.items() if not s.get('is_script', False)]
    if len(valid_keys) < 2:
        return 0.0
    consensuses = [row_signals[j].get('consensus', 0.5) for j in valid_keys]
    return 1.0 - np.mean(consensuses)

sample_diversity = {aid: compute_sample_diversity(sigs) for aid, sigs in signal_data.items()}
sample_regime = {aid: ('diverse' if d >= DIVERSITY_THRESHOLD else 'similar')
                 for aid, d in sample_diversity.items()}

div_values = list(sample_diversity.values())
regime_counts = pd.Series(sample_regime).value_counts()

print("=" * 60)
print("REGIME DETECTION")
print("=" * 60)
print(f"Threshold: {DIVERSITY_THRESHOLD}")
print(f"Diversity: mean={np.mean(div_values):.3f}, min={np.min(div_values):.3f}, "
      f"max={np.max(div_values):.3f}")
print(f"Similar: {regime_counts.get('similar', 0)}, Diverse: {regime_counts.get('diverse', 0)}")

# Split labeled data
labeled['diversity'] = labeled['audio_id'].map(sample_diversity)
labeled['regime'] = labeled['audio_id'].map(sample_regime)
labeled_similar = labeled[labeled['regime'] == 'similar'].copy()
labeled_diverse = labeled[labeled['regime'] == 'diverse'].copy()

print(f"\nLabeled similar: {len(labeled_similar)}, Labeled diverse: {len(labeled_diverse)}")

# Auto-adjust threshold if one group is empty
if len(labeled_similar) == 0 or len(labeled_diverse) == 0:
    print(f"One group empty - adjusting threshold to median...")
    DIVERSITY_THRESHOLD = float(np.median(labeled['diversity']))
    labeled['regime'] = labeled['diversity'].apply(
        lambda d: 'diverse' if d >= DIVERSITY_THRESHOLD else 'similar')
    sample_regime = {aid: ('diverse' if d >= DIVERSITY_THRESHOLD else 'similar')
                     for aid, d in sample_diversity.items()}
    labeled_similar = labeled[labeled['regime'] == 'similar'].copy()
    labeled_diverse = labeled[labeled['regime'] == 'diverse'].copy()
    print(f"  New threshold: {DIVERSITY_THRESHOLD:.4f}")
    print(f"  Similar: {len(labeled_similar)}, Diverse: {len(labeled_diverse)}")

In [ ]:
# ============================================================
# Cell 11: Scoring Functions & Baselines
# ============================================================

def pick_best_option(row_signals, weights, use_hard_filter=True):
    """Score options and pick the best one."""
    candidates = {}
    for j, sig in row_signals.items():
        if sig.get('is_script', False): continue
        if use_hard_filter and sig.get('hard_filter', False): continue
        score = (
            weights.get('whisper', 0) * sig.get('whisper_sim', 0) +
            weights.get('seamless', 0) * sig.get('seamless_sim', 0) +
            weights.get('consensus', 0) * sig.get('consensus', 0) +
            weights.get('fluency', 0) * sig.get('fluency', 0) -
            weights.get('quality', 0) * sig.get('quality_penalty', 0) -
            weights.get('length', 0) * sig.get('rel_length', 0.5)
        )
        candidates[j] = score
    if not candidates and use_hard_filter:
        return pick_best_option(row_signals, weights, use_hard_filter=False)
    if not candidates:
        return 1
    return max(candidates, key=candidates.get)


def pick_best_option_adaptive(row_signals, weights_similar, weights_diverse,
                               diversity, threshold, use_hard_filter=True):
    """Adaptive: choose weight set based on sample diversity."""
    w = weights_diverse if diversity >= threshold else weights_similar
    return pick_best_option(row_signals, w, use_hard_filter)


def evaluate_weights(weights, labeled_df, signal_data, use_hard_filter=True):
    correct = total = 0
    for _, row in labeled_df.iterrows():
        aid = row['audio_id']
        if aid not in signal_data: continue
        pred = pick_best_option(signal_data[aid], weights, use_hard_filter)
        if pred == int(row['correct_int']): correct += 1
        total += 1
    return correct, total


def evaluate_adaptive(w_sim, w_div, labeled_df, signal_data, sample_diversity, threshold):
    correct = total = 0
    rc = {'similar': 0, 'diverse': 0}
    rt = {'similar': 0, 'diverse': 0}
    for _, row in labeled_df.iterrows():
        aid = row['audio_id']
        if aid not in signal_data: continue
        actual = int(row['correct_int'])
        div = sample_diversity.get(aid, 0.0)
        regime = 'diverse' if div >= threshold else 'similar'
        pred = pick_best_option_adaptive(signal_data[aid], w_sim, w_div, div, threshold)
        if pred == actual:
            correct += 1
            rc[regime] += 1
        total += 1
        rt[regime] += 1
    return correct, total, rc, rt


# ---- Baselines ----
print("=" * 60)
print("BASELINES")
print("=" * 60)
for name, w in [
    ("Whisper only", {'whisper': 1, 'seamless': 0, 'consensus': 0, 'fluency': 0, 'quality': 0, 'length': 0}),
    ("Seamless only", {'whisper': 0, 'seamless': 1, 'consensus': 0, 'fluency': 0, 'quality': 0, 'length': 0}),
    ("Neg consensus", {'whisper': 0, 'seamless': 0, 'consensus': -1, 'fluency': 0, 'quality': 0, 'length': 0}),
]:
    c, t = evaluate_weights(w, labeled, signal_data)
    print(f"  {name:20s}: {c}/{t} = {c/t*100:.1f}%")

In [ ]:
# ============================================================
# Cell 12: Two-Regime Grid Search
# ============================================================

grid = {
    'whisper':   [0.0, 0.1, 0.2, 0.3, 0.5, 0.7],
    'seamless':  [0.0, 0.1, 0.2, 0.3, 0.5, 0.7],
    'consensus': [-0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2],
    'fluency':   [-0.3, -0.2, -0.1, 0.0],
    'quality':   [0.0, 0.1, 0.2, 0.3, 0.5],
    'length':    [0.0, 0.05, 0.1],
}

keys = list(grid.keys())
value_lists = [grid[k] for k in keys]
total_combos = 1
for v in grid.values(): total_combos *= len(v)
print(f"Total weight combinations: {total_combos:,}")

# ---- SIMILAR regime ----
print(f"\n{'='*60}")
print(f"GRID SEARCH - SIMILAR REGIME ({len(labeled_similar)} samples)")
print("="*60)

best_similar_acc = 0
best_similar_weights = None
best_similar_correct = 0

if len(labeled_similar) > 0:
    for combo in tqdm(iter_product(*value_lists), total=total_combos, desc="Similar"):
        weights = dict(zip(keys, combo))
        if weights['whisper'] == 0 and weights['seamless'] == 0 and weights['consensus'] == 0:
            continue
        c, t = evaluate_weights(weights, labeled_similar, signal_data)
        acc = c / t if t > 0 else 0
        if acc > best_similar_acc or (acc == best_similar_acc and
            sum(abs(v) for v in weights.values()) < sum(abs(v) for v in (best_similar_weights or weights).values())):
            best_similar_acc = acc
            best_similar_weights = weights.copy()
            best_similar_correct = c
    print(f"Best SIMILAR: {best_similar_correct}/{len(labeled_similar)} = {best_similar_acc*100:.1f}%")
    print(f"Weights: {best_similar_weights}")
else:
    best_similar_weights = {'whisper': 0, 'seamless': 0.1, 'consensus': -0.3, 'fluency': -0.2, 'quality': 0.1, 'length': 0}
    print(f"No similar samples - using default weights: {best_similar_weights}")

# ---- DIVERSE regime ----
print(f"\n{'='*60}")
print(f"GRID SEARCH - DIVERSE REGIME ({len(labeled_diverse)} samples)")
print("="*60)

best_diverse_acc = 0
best_diverse_weights = None
best_diverse_correct = 0

if len(labeled_diverse) > 0:
    for combo in tqdm(iter_product(*value_lists), total=total_combos, desc="Diverse"):
        weights = dict(zip(keys, combo))
        if weights['whisper'] == 0 and weights['seamless'] == 0 and weights['consensus'] == 0:
            continue
        c, t = evaluate_weights(weights, labeled_diverse, signal_data)
        acc = c / t if t > 0 else 0
        if acc > best_diverse_acc or (acc == best_diverse_acc and
            sum(abs(v) for v in weights.values()) < sum(abs(v) for v in (best_diverse_weights or weights).values())):
            best_diverse_acc = acc
            best_diverse_weights = weights.copy()
            best_diverse_correct = c
    print(f"Best DIVERSE: {best_diverse_correct}/{len(labeled_diverse)} = {best_diverse_acc*100:.1f}%")
    print(f"Weights: {best_diverse_weights}")
else:
    best_diverse_weights = {'whisper': 0, 'seamless': 0.3, 'consensus': 0.2, 'fluency': 0, 'quality': 0, 'length': 0.05}
    print(f"No diverse samples - using default weights: {best_diverse_weights}")

# ---- SINGLE regime (v2 baseline) ----
print(f"\n{'='*60}")
print(f"GRID SEARCH - SINGLE REGIME (all {len(labeled)} labeled)")
print("="*60)

best_single_acc = 0
best_single_weights = None
best_single_correct = 0

for combo in tqdm(iter_product(*value_lists), total=total_combos, desc="Single"):
    weights = dict(zip(keys, combo))
    if weights['whisper'] == 0 and weights['seamless'] == 0 and weights['consensus'] == 0:
        continue
    c, t = evaluate_weights(weights, labeled, signal_data)
    acc = c / t
    if acc > best_single_acc or (acc == best_single_acc and
        sum(abs(v) for v in weights.values()) < sum(abs(v) for v in (best_single_weights or weights).values())):
        best_single_acc = acc
        best_single_weights = weights.copy()
        best_single_correct = c

print(f"Best SINGLE: {best_single_correct}/{len(labeled)} = {best_single_acc*100:.1f}%")
print(f"Weights: {best_single_weights}")

# ---- Combined adaptive evaluation ----
print(f"\n{'='*60}")
print("ADAPTIVE vs SINGLE")
print("="*60)
c, t, rc, rt = evaluate_adaptive(
    best_similar_weights, best_diverse_weights,
    labeled, signal_data, sample_diversity, DIVERSITY_THRESHOLD
)
print(f"Adaptive: {c}/{t} = {c/t*100:.1f}%")
for reg in ['similar', 'diverse']:
    if rt[reg] > 0:
        print(f"  {reg}: {rc[reg]}/{rt[reg]} = {rc[reg]/rt[reg]*100:.1f}%")
print(f"Single:   {best_single_correct}/{len(labeled)} = {best_single_acc*100:.1f}%")

In [ ]:
# ============================================================
# Cell 13: Leave-One-Out Cross-Validation
# ============================================================
print("=" * 60)
print("LEAVE-ONE-OUT CROSS-VALIDATION")
print("=" * 60)

# LOO-CV Adaptive
loo_adaptive_correct = 0
loo_details = []
for _, test_row in labeled.iterrows():
    aid = test_row['audio_id']
    actual = int(test_row['correct_int'])
    if aid not in signal_data: continue
    div = sample_diversity.get(aid, 0.0)
    pred = pick_best_option_adaptive(
        signal_data[aid], best_similar_weights, best_diverse_weights,
        div, DIVERSITY_THRESHOLD
    )
    regime = 'diverse' if div >= DIVERSITY_THRESHOLD else 'similar'
    is_correct = pred == actual
    if is_correct: loo_adaptive_correct += 1
    loo_details.append({'audio_id': aid, 'regime': regime, 'correct': is_correct})

loo_adaptive_acc = loo_adaptive_correct / len(labeled)

# LOO-CV Single
loo_single_correct = 0
for _, test_row in labeled.iterrows():
    aid = test_row['audio_id']
    actual = int(test_row['correct_int'])
    if aid in signal_data:
        pred = pick_best_option(signal_data[aid], best_single_weights)
        if pred == actual: loo_single_correct += 1

loo_single_acc = loo_single_correct / len(labeled)

print(f"Adaptive (two-regime): {loo_adaptive_correct}/{len(labeled)} = {loo_adaptive_acc*100:.1f}%")
print(f"Single-regime (v2):    {loo_single_correct}/{len(labeled)} = {loo_single_acc*100:.1f}%")

loo_df = pd.DataFrame(loo_details)
for regime in ['similar', 'diverse']:
    sub = loo_df[loo_df['regime'] == regime]
    if len(sub) > 0:
        print(f"  {regime}: {sub['correct'].sum()}/{len(sub)} = {sub['correct'].mean()*100:.1f}%")

# Auto-select best system
if loo_adaptive_correct >= loo_single_correct:
    USE_ADAPTIVE = True
    FINAL_WEIGHTS_SIMILAR = best_similar_weights
    FINAL_WEIGHTS_DIVERSE = best_diverse_weights
    FINAL_WEIGHTS_SINGLE = best_single_weights
    print(f"\n>>> SELECTED: Adaptive two-regime ({loo_adaptive_acc*100:.1f}%)")
    print(f"    Similar weights: {FINAL_WEIGHTS_SIMILAR}")
    print(f"    Diverse weights: {FINAL_WEIGHTS_DIVERSE}")
else:
    USE_ADAPTIVE = False
    FINAL_WEIGHTS_SINGLE = best_single_weights
    FINAL_WEIGHTS_SIMILAR = best_similar_weights
    FINAL_WEIGHTS_DIVERSE = best_diverse_weights
    print(f"\n>>> SELECTED: Single-regime ({loo_single_acc*100:.1f}%)")
    print(f"    Weights: {FINAL_WEIGHTS_SINGLE}")

In [ ]:
# ============================================================
# Cell 14: Error Analysis
# ============================================================
print("=" * 60)
print("VALIDATION ON LABELED DATA")
print("=" * 60)

val_correct = 0
val_misses = []

for _, row in labeled.iterrows():
    aid = row['audio_id']
    actual = int(row['correct_int'])
    div = sample_diversity.get(aid, 0.0)
    
    if USE_ADAPTIVE:
        pred = pick_best_option_adaptive(
            signal_data[aid], FINAL_WEIGHTS_SIMILAR, FINAL_WEIGHTS_DIVERSE,
            div, DIVERSITY_THRESHOLD)
    else:
        pred = pick_best_option(signal_data[aid], FINAL_WEIGHTS_SINGLE)
    
    if pred == actual:
        val_correct += 1
    else:
        p_sig = signal_data[aid].get(pred, {})
        a_sig = signal_data[aid].get(actual, {})
        val_misses.append({
            'audio_id': aid, 'predicted': pred, 'actual': actual,
            'diversity': div,
            'pred_consensus': p_sig.get('consensus', 0),
            'actual_consensus': a_sig.get('consensus', 0),
            'pred_seamless': p_sig.get('seamless_sim', 0),
            'actual_seamless': a_sig.get('seamless_sim', 0),
        })

print(f"ACCURACY: {val_correct}/{len(labeled)} = {val_correct/len(labeled)*100:.1f}%")

if val_misses:
    print(f"\nMisclassified ({len(val_misses)}):\n")
    for m in val_misses:
        print(f"  id={m['audio_id']}: pred={m['predicted']} actual={m['actual']} div={m['diversity']:.4f}")
        print(f"    consensus: pred={m['pred_consensus']:.3f} vs actual={m['actual_consensus']:.3f}")
        print(f"    seamless:  pred={m['pred_seamless']:.3f} vs actual={m['actual_seamless']:.3f}")
else:
    print("\nPERFECT - no misclassified samples!")

In [ ]:
# ============================================================
# Cell 15: Generate Predictions for ALL 100 Samples
# ============================================================
print(f"Generating predictions for all {len(df)} samples...")

final_predictions = {}
for _, row in df.iterrows():
    aid = row['audio_id']
    if aid not in signal_data:
        final_predictions[aid] = 1
        continue
    div = sample_diversity.get(aid, 0.0)
    if USE_ADAPTIVE:
        final_predictions[aid] = pick_best_option_adaptive(
            signal_data[aid], FINAL_WEIGHTS_SIMILAR, FINAL_WEIGHTS_DIVERSE,
            div, DIVERSITY_THRESHOLD)
    else:
        final_predictions[aid] = pick_best_option(signal_data[aid], FINAL_WEIGHTS_SINGLE)

df['golden_option'] = df['audio_id'].map(final_predictions)
print(f"Prediction distribution: {df['golden_option'].value_counts().sort_index().to_dict()}")

regime_pred = {aid: ('diverse' if sample_diversity.get(aid, 0) >= DIVERSITY_THRESHOLD else 'similar')
               for aid in final_predictions}
print(f"Regime distribution: {pd.Series(regime_pred).value_counts().to_dict()}")

In [ ]:
# ============================================================
# Cell 16: Build & Save Output CSV
# ============================================================
print("Building output CSV...")
output_rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Output"):
    aid = row['audio_id']
    golden_opt = int(final_predictions[aid])
    golden_text = str(row[f'option_{golden_opt}'])
    
    result = {
        'audio_id': aid,
        'language': row['language'],
        'audio': row['audio'],
        'option_1': str(row['option_1']),
        'option_2': str(row['option_2']),
        'option_3': str(row['option_3']),
        'option_4': str(row['option_4']),
        'option_5': str(row['option_5']),
        'golden_ref': f'option_{golden_opt}',
        'regime': regime_pred.get(aid, 'unknown'),
        'diversity': round(sample_diversity.get(aid, 0.0), 4),
    }
    
    # WER of each option vs golden
    for j in range(1, 6):
        opt_text = str(row[f'option_{j}'])
        if is_script_option(opt_text) or len(opt_text.strip()) < 2:
            result[f'wer_option{j}'] = 1.0
        elif j == golden_opt:
            result[f'wer_option{j}'] = 0.0
        else:
            result[f'wer_option{j}'] = round(compute_wer_safe(golden_text, opt_text), 4)
    
    # Per-option signal scores
    sigs = signal_data.get(aid, {})
    for j in range(1, 6):
        s = sigs.get(j, {})
        result[f'whisper_sim_option{j}'] = round(s.get('whisper_sim', 0), 4)
        result[f'seamless_sim_option{j}'] = round(s.get('seamless_sim', 0), 4)
        result[f'consensus_option{j}'] = round(s.get('consensus', 0), 4)
        result[f'quality_penalty_option{j}'] = round(s.get('quality_penalty', 0), 4)
        result[f'hard_filtered_option{j}'] = s.get('hard_filter', True)
    
    result['source'] = 'pipeline_v5'
    output_rows.append(result)

output_df = pd.DataFrame(output_rows)
output_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\nOutput saved: {OUTPUT_CSV}")
print(f"Shape: {output_df.shape}")
print(f"\nGolden ref distribution:")
print(output_df['golden_ref'].value_counts().sort_index())

# Also save detailed signal scores
detailed_path = OUTPUT_CSV.replace('.csv', '_detailed.csv')
detail_rows = []
for aid, sigs in signal_data.items():
    for j, s in sigs.items():
        if s.get('is_script', False): continue
        detail_rows.append({
            'audio_id': aid, 'option': j,
            'whisper_sim': round(s.get('whisper_sim', 0), 4),
            'seamless_sim': round(s.get('seamless_sim', 0), 4),
            'consensus': round(s.get('consensus', 0), 4),
            'fluency': round(s.get('fluency', 0), 4),
            'quality_penalty': round(s.get('quality_penalty', 0), 4),
            'hard_filter': s.get('hard_filter', True),
            'regime': sample_regime.get(aid, 'unknown'),
            'diversity': round(sample_diversity.get(aid, 0.0), 4),
            'is_golden': j == final_predictions.get(aid, -1),
        })
pd.DataFrame(detail_rows).to_csv(detailed_path, index=False)
print(f"Detailed scores: {detailed_path}")

In [ ]:
# ============================================================
# Cell 17: Visualization
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle('Golden Transcription Selection v5 - Arabic_SA', fontsize=15, fontweight='bold')

# (a) Golden option distribution
ax = axes[0, 0]
df['golden_option'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('(a) Golden Option Distribution')
ax.set_xlabel('Option'); ax.set_ylabel('Count')

# (b) Diversity histogram
ax = axes[0, 1]
divs = [sample_diversity[aid] for aid in df['audio_id']]
ax.hist(divs, bins=20, color='mediumpurple', edgecolor='black', alpha=0.7)
ax.axvline(DIVERSITY_THRESHOLD, color='red', linestyle='--', linewidth=2,
           label=f'Threshold={DIVERSITY_THRESHOLD:.3f}')
ax.set_title('(b) Diversity Distribution'); ax.set_xlabel('Diversity'); ax.legend()

# (c) Per-regime accuracy
ax = axes[0, 2]
regimes_list = ['Similar', 'Diverse', 'Overall']
adap_accs, single_accs = [], []
for sub_df in [labeled_similar, labeled_diverse, labeled]:
    if len(sub_df) > 0:
        ca = cs = 0
        for _, r in sub_df.iterrows():
            aid, actual = r['audio_id'], int(r['correct_int'])
            d = sample_diversity.get(aid, 0)
            if pick_best_option_adaptive(signal_data[aid], best_similar_weights, best_diverse_weights, d, DIVERSITY_THRESHOLD) == actual: ca += 1
            if pick_best_option(signal_data[aid], best_single_weights) == actual: cs += 1
        adap_accs.append(ca/len(sub_df)*100); single_accs.append(cs/len(sub_df)*100)
    else:
        adap_accs.append(0); single_accs.append(0)
x = np.arange(len(regimes_list))
ax.bar(x-0.2, adap_accs, 0.35, label='Adaptive', color='green', alpha=0.7)
ax.bar(x+0.2, single_accs, 0.35, label='Single', color='orange', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(regimes_list)
ax.set_title('(c) Adaptive vs Single'); ax.set_ylabel('Accuracy %'); ax.legend()

# (d) Weight comparison
ax = axes[1, 0]
w_names = list(best_similar_weights.keys())
x = np.arange(len(w_names))
ax.bar(x-0.2, [best_similar_weights[k] for k in w_names], 0.35, label='Similar', color='skyblue')
ax.bar(x+0.2, [best_diverse_weights[k] for k in w_names], 0.35, label='Diverse', color='coral')
ax.set_xticks(x); ax.set_xticklabels(w_names, rotation=30, fontsize=9)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('(d) Weights: Similar vs Diverse'); ax.legend()

# (e) Signal comparison: golden vs non-golden
ax = axes[1, 1]
detail_df = pd.DataFrame(detail_rows)
sig_names = ['whisper_sim', 'seamless_sim', 'consensus', 'quality_penalty']
golden_m = [detail_df[detail_df['is_golden']][s].mean() for s in sig_names]
nongolden_m = [detail_df[~detail_df['is_golden']][s].mean() for s in sig_names]
x = np.arange(len(sig_names))
ax.bar(x-0.2, golden_m, 0.35, label='Golden', color='green', alpha=0.7)
ax.bar(x+0.2, nongolden_m, 0.35, label='Non-Golden', color='red', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(sig_names, rotation=30, fontsize=9)
ax.set_title('(e) Signal: Golden vs Non-Golden'); ax.legend()

# (f) Diversity vs correctness
ax = axes[1, 2]
for idx, (_, r) in enumerate(labeled.iterrows()):
    aid = r['audio_id']
    d = sample_diversity.get(aid, 0)
    correct = final_predictions.get(aid) == int(r['correct_int'])
    ax.scatter(d, idx, c='green' if correct else 'red',
               marker='o' if d < DIVERSITY_THRESHOLD else 's', s=40, alpha=0.7)
ax.axvline(DIVERSITY_THRESHOLD, color='blue', linestyle='--', alpha=0.5)
ax.set_xlabel('Diversity'); ax.set_ylabel('Sample Index')
ax.set_title('(f) Diversity vs Correctness')

plt.tight_layout()
plt.savefig('/kaggle/working/analysis_v5_arabic_sa.png', dpi=150, bbox_inches='tight')
plt.show()
print("Analysis panel saved.")

In [ ]:
# ============================================================
# Cell 18: Final Summary
# ============================================================
print("\n" + "=" * 60)
print("PIPELINE v5 - ARABIC_SA SUBMISSION SUMMARY")
print("=" * 60)
print(f"Dataset:             Arabic_SA ({len(df)} samples, {len(labeled)} labeled)")
print(f"ASR Models:          Whisper large-v3 + SeamlessM4T v2-large")
print(f"Mode:                {'Adaptive two-regime' if USE_ADAPTIVE else 'Single-regime'}")
print(f"Diversity Threshold: {DIVERSITY_THRESHOLD:.4f}")
print(f"")
if USE_ADAPTIVE:
    print(f"Similar Weights:     {FINAL_WEIGHTS_SIMILAR}")
    print(f"Diverse Weights:     {FINAL_WEIGHTS_DIVERSE}")
else:
    print(f"Single Weights:      {FINAL_WEIGHTS_SINGLE}")
print(f"")
print(f"Validation Accuracy: {val_correct}/{len(labeled)} = {val_correct/len(labeled)*100:.1f}%")
print(f"LOO-CV (adaptive):   {loo_adaptive_correct}/{len(labeled)} = {loo_adaptive_acc*100:.1f}%")
print(f"LOO-CV (single):     {loo_single_correct}/{len(labeled)} = {loo_single_acc*100:.1f}%")
print(f"Total predictions:   {len(final_predictions)}")
print(f"Output CSV:          {OUTPUT_CSV}")
print(f"")
print(f"Golden ref distribution:")
print(output_df['golden_ref'].value_counts().sort_index().to_string())
print(f"")
print(f"Regime distribution:")
print(f"  Similar: {sum(1 for v in regime_pred.values() if v == 'similar')}")
print(f"  Diverse: {sum(1 for v in regime_pred.values() if v == 'diverse')}")
print("=" * 60)
print("SUBMISSION READY")
print("=" * 60)